In [1]:
import pandas as pd
import numpy as np

# STRING

In [2]:
hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

string = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.links.detailed.v12.0.txt',sep=' ')
string_info = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.info.v12.0.txt',sep='\t')

string['protein1'] = string['protein1'].str.replace('9606.', '')
string['protein2'] = string['protein2'].str.replace('9606.', '')
string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
string_info = string_info.iloc[:, :2]

string = string.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_1'})
string = string.drop('#string_protein_id', axis=1)

# Repeat for 'protein2' column
string = string.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_2'})
string = string.drop('#string_protein_id', axis=1)


# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))


# Ensure all entries are treated as strings
string['gene_symbol_1'] = string['gene_symbol_1'].astype(str)
string['gene_symbol_2'] = string['gene_symbol_2'].astype(str)

# Apply the lookup function to both gene_symbol_1 and gene_symbol_2 columns
string['hgnc_id_1'], string['ensembl_gene_id_1'] = zip(*string['gene_symbol_1'].apply(lookup_gene_info))
string['hgnc_id_2'], string['ensembl_gene_id_2'] = zip(*string['gene_symbol_2'].apply(lookup_gene_info))



/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_23611/2599185819.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')


In [3]:
def run_string_biomarker_query_for_pair(main_csv_path: str, string_df: pd.DataFrame, output_prefix: str):
    """
    Compute STRING interaction scores for biomarker–query pairs in a given SL dataset.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATR_ATM_main.csv).
    string_df : pd.DataFrame
        STRING interactions with columns ['ensembl_gene_id_1', 'ensembl_gene_id_2', 'combined_score'].
    output_prefix : str
        Prefix for output file (e.g. "ATR_ATM").

    Returns
    -------
    string_biomarker_query : pd.DataFrame
        DataFrame containing query, biomarker, and STRING score.
    """
    # Load SL dataset
    main_csv = pd.read_csv(main_csv_path, low_memory=False)

    # Build STRING dictionary (both directions)
    string_dict = {
        (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
        for _, row in string_df.iterrows()
    }
    string_dict.update({
        (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
        for _, row in string_df.iterrows()
    })

    # Lookup function
    def get_combined_score(row):
        pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_biomarker'])
        pair2 = (row['ensembl_gene_id_biomarker'], row['ensembl_gene_id_query'])  # reverse
        return string_dict.get(pair1) or string_dict.get(pair2)

    # Apply STRING scores
    main_csv['StringInteractionWithBiomarker'] = main_csv.apply(get_combined_score, axis=1)

    # Save results
    string_biomarker_query = main_csv[['ensembl_gene_id_query', 'ensembl_gene_id_biomarker', 'StringInteractionWithBiomarker']]
    string_biomarker_query.to_csv(
        f'feature_output/string_score_query_biomarker_{output_prefix}.csv',
        index=False
    )

    print(f"✅ STRING biomarker–query scores saved for {output_prefix}")
    return string_biomarker_query


In [4]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
pairs_df = pd.read_excel(excel_file)

# Assume STRING dataframe already loaded once
# string = pd.read_csv("path_to_string_file.txt", sep="\t")

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_string_biomarker_query_for_pair(
        main_csv_path=main_file,
        string_df=string,
        output_prefix=SL_pair
    )


✅ STRING biomarker–query scores saved for MSH6_WRN
✅ STRING biomarker–query scores saved for CDH1_ROS1
✅ STRING biomarker–query scores saved for ATM_ATR
✅ STRING biomarker–query scores saved for BRCA1_ATR
✅ STRING biomarker–query scores saved for CCNE1_PKMYT1
✅ STRING biomarker–query scores saved for FBXW7_PKMYT1
✅ STRING biomarker–query scores saved for BRCA1_PARG
✅ STRING biomarker–query scores saved for BRCA1_USP1
✅ STRING biomarker–query scores saved for BRCA1_POLQ
✅ STRING biomarker–query scores saved for KRAS_PLK1


In [5]:
def run_string_target1_query_for_pair(main_csv_path: str, string_df: pd.DataFrame, output_prefix: str):
    """
    Compute STRING interaction scores for target1–query pairs in a given SL dataset.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATR_ATM_main.csv).
    string_df : pd.DataFrame
        STRING interactions with columns ['ensembl_gene_id_1', 'ensembl_gene_id_2', 'combined_score'].
    output_prefix : str
        Prefix for output file (e.g. "ATR_ATM").

    Returns
    -------
    string_target1_query : pd.DataFrame
        DataFrame containing query, target1, and STRING score.
    """
    # Load SL dataset
    main_csv = pd.read_csv(main_csv_path, low_memory=False)

    # Build STRING dictionary (both directions)
    string_dict = {
        (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
        for _, row in string_df.iterrows()
    }
    string_dict.update({
        (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
        for _, row in string_df.iterrows()
    })

    # Lookup function
    def get_combined_score(row):
        pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target1'])
        pair2 = (row['ensembl_gene_id_target1'], row['ensembl_gene_id_query'])  # reverse
        return string_dict.get(pair1) or string_dict.get(pair2)

    # Apply STRING scores
    main_csv['StringInteractionWithTarget'] = main_csv.apply(get_combined_score, axis=1)

    # Save results
    string_target1_query = main_csv[['ensembl_gene_id_query', 'ensembl_gene_id_target1', 'StringInteractionWithTarget']]
    string_target1_query.to_csv(
        f'feature_output/string_score_query_target1_{output_prefix}.csv',
        index=False
    )

    print(f"✅ STRING target1–query scores saved for {output_prefix}")
    return string_target1_query


In [6]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
pairs_df = pd.read_excel(excel_file)

# Assume STRING dataframe already loaded once
# string = pd.read_csv("path_to_string_file.txt", sep="\t")

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_string_target1_query_for_pair(
        main_csv_path=main_file,
        string_df=string,
        output_prefix=SL_pair
    )


✅ STRING target1–query scores saved for MSH6_WRN
✅ STRING target1–query scores saved for CDH1_ROS1
✅ STRING target1–query scores saved for ATM_ATR
✅ STRING target1–query scores saved for BRCA1_ATR
✅ STRING target1–query scores saved for CCNE1_PKMYT1
✅ STRING target1–query scores saved for FBXW7_PKMYT1
✅ STRING target1–query scores saved for BRCA1_PARG
✅ STRING target1–query scores saved for BRCA1_USP1
✅ STRING target1–query scores saved for BRCA1_POLQ
✅ STRING target1–query scores saved for KRAS_PLK1


# BIOGRID

In [7]:
hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

# Sort values in each row and create a new column containing sorted combinations
biogrid['SortedInteractors'] = biogrid.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)

# Drop duplicates based on the new column
biogrid.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)

# Drop the auxiliary column
biogrid.drop(columns='SortedInteractors', inplace=True)

biogrid= biogrid.reset_index(drop=True)

# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan))



# Ensure all entries are treated as strings
biogrid['Official Symbol Interactor A'] = biogrid['Official Symbol Interactor A'].astype(str)
biogrid['Official Symbol Interactor B'] = biogrid['Official Symbol Interactor B'].astype(str)

# Apply the lookup function to both gene_symbol_1 and gene_symbol_2 columns
biogrid['hgnc_id_1'], biogrid['ensembl_gene_id_1'] = zip(*biogrid['Official Symbol Interactor A'].apply(lookup_gene_info))
biogrid['hgnc_id_2'], biogrid['ensembl_gene_id_2'] = zip(*biogrid['Official Symbol Interactor B'].apply(lookup_gene_info))
biogrid

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_23611/1183806919.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_23611/1183806919.py:5: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


,#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,...,Ontology Term Categories,Ontology Term Qualifier IDs,Ontology Term Qualifier Names,Ontology Term Types,Organism Name Interactor A,Organism Name Interactor B,hgnc_id_1,ensembl_gene_id_1,hgnc_id_2,ensembl_gene_id_2
0,103,6416,2318,112315,108607,-,-,MAP2K4,FLNC,JNKK|JNKK1|MAPKK4|MEK4|MKK4|PRKMK4|SAPKK-1|SAP...,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:6844,ENSG00000065559,HGNC:3756,ENSG00000128591
1,278,2624,5371,108894,111384,-,-,GATA2,PML,DCML|IMD21|MONOMAC|NFE1B,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:4171,ENSG00000179348,HGNC:9113,ENSG00000140464
2,418,6118,6774,112038,112651,RP4-547C9.3,-,RPA2,STAT3,REPA2|RP-A p32|RP-A p34|RPA32,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:10290,ENSG00000117748,HGNC:11364,ENSG00000168610
3,586,375,23163,106870,116775,-,-,ARF1,GGA3,-,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:652,ENSG00000143761,HGNC:17079,ENSG00000125447
4,612,377,23647,106872,117174,-,-,ARF3,ARFIP2,-,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:654,ENSG00000134287,HGNC:17160,ENSG00000132254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84513,3629930,582,9867,107058,115200,-,-,BBS1,PJA2,BBS2L2,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:966,ENSG00000174483,HGNC:17481,ENSG00000198961
84514,3629941,8975,7291,114465,113142,-,-,USP13,TWIST1,ISOT3|IsoT-3,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:12611,ENSG00000058056,HGNC:12428,ENSG00000122691
84515,3629949,5562,1317,111549,107712,-,RP11-110L1.1,PRKAA1,SLC31A1,AMPK|AMPKa1,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:9376,ENSG00000132356,HGNC:11016,ENSG00000136868
84516,3638506,5566,2534,111553,108810,-,RP1-66H14.1,PRKACA,FYN,PKACA,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:9380,ENSG00000072062,HGNC:4037,ENSG00000010810


In [8]:
def run_biogrid_biomarker_query_for_pair(main_csv_path: str, biogrid_df: pd.DataFrame, output_prefix: str):
    """
    Compute BioGRID physical interaction (yes/no) for biomarker–query pairs 
    in a given SL dataset.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATR_ATM_main.csv).
    biogrid_df : pd.DataFrame
        BioGRID interactions with columns ['ensembl_gene_id_1', 'ensembl_gene_id_2'].
    output_prefix : str
        Prefix for output file (e.g. "ATR_ATM").

    Returns
    -------
    biogrid_query_biomarker : pd.DataFrame
        DataFrame containing biomarker–query BioGRID interaction presence.
    """
    # Load SL dataset
    main_csv = pd.read_csv(main_csv_path, low_memory=False)

    # Build BioGRID dictionary (undirected edges)
    biogrid_dict = {
        (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
        for _, row in biogrid_df.iterrows()
    }
    biogrid_dict.update({
        (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
        for _, row in biogrid_df.iterrows()
    })

    # Lookup function
    def check_biogrid_interaction(row):
        pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_biomarker'])
        pair2 = (row['ensembl_gene_id_biomarker'], row['ensembl_gene_id_query'])  # reverse
        return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

    # Apply BioGRID interaction check
    main_csv['BIOGRIDPhysicalInteractionQueryBiomarker'] = main_csv.apply(check_biogrid_interaction, axis=1)

    # Save results
    biogrid_query_biomarker = main_csv[
        ['ensembl_gene_id_query', 'ensembl_gene_id_biomarker', 'BIOGRIDPhysicalInteractionQueryBiomarker']
    ]
    biogrid_query_biomarker.to_csv(
        f'feature_output/biogrid_biomarker_query_{output_prefix}.csv',
        index=False
    )

    print(f"✅ BioGRID biomarker–query interactions saved for {output_prefix}")
    return biogrid_query_biomarker


In [9]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
pairs_df = pd.read_excel(excel_file)

# Assume BioGRID dataframe already loaded once
# biogrid = pd.read_csv("path_to_biogrid_file.txt", sep="\t")

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_biogrid_biomarker_query_for_pair(
        main_csv_path=main_file,
        biogrid_df=biogrid,
        output_prefix=SL_pair
    )


✅ BioGRID biomarker–query interactions saved for MSH6_WRN
✅ BioGRID biomarker–query interactions saved for CDH1_ROS1
✅ BioGRID biomarker–query interactions saved for ATM_ATR
✅ BioGRID biomarker–query interactions saved for BRCA1_ATR
✅ BioGRID biomarker–query interactions saved for CCNE1_PKMYT1
✅ BioGRID biomarker–query interactions saved for FBXW7_PKMYT1
✅ BioGRID biomarker–query interactions saved for BRCA1_PARG
✅ BioGRID biomarker–query interactions saved for BRCA1_USP1
✅ BioGRID biomarker–query interactions saved for BRCA1_POLQ
✅ BioGRID biomarker–query interactions saved for KRAS_PLK1


In [10]:
def run_biogrid_target1_query_for_pair(main_csv_path: str, biogrid_df: pd.DataFrame, output_prefix: str):
    """
    Compute BioGRID physical interaction (yes/no) for target1–query pairs 
    in a given SL dataset.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATR_ATM_main.csv).
    biogrid_df : pd.DataFrame
        BioGRID interactions with columns ['ensembl_gene_id_1', 'ensembl_gene_id_2'].
    output_prefix : str
        Prefix for output file (e.g. "ATR_ATM").

    Returns
    -------
    biogrid_query_target : pd.DataFrame
        DataFrame containing target1–query BioGRID interaction presence.
    """
    # Load SL dataset
    main_csv = pd.read_csv(main_csv_path, low_memory=False)

    # Build BioGRID dictionary (undirected edges)
    biogrid_dict = {
        (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
        for _, row in biogrid_df.iterrows()
    }
    biogrid_dict.update({
        (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
        for _, row in biogrid_df.iterrows()
    })

    # Lookup function
    def check_biogrid_interaction(row):
        pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target1'])
        pair2 = (row['ensembl_gene_id_target1'], row['ensembl_gene_id_query'])  # reverse
        return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

    # Apply BioGRID interaction check
    main_csv['BIOGRIDPhysicalInteractionQueryTarget1'] = main_csv.apply(check_biogrid_interaction, axis=1)

    # Save results
    biogrid_query_target = main_csv[
        ['ensembl_gene_id_query', 'ensembl_gene_id_target1', 'BIOGRIDPhysicalInteractionQueryTarget1']
    ]
    biogrid_query_target.to_csv(
        f'feature_output/biogrid_target1_query_{output_prefix}.csv',
        index=False
    )

    print(f"✅ BioGRID target1–query interactions saved for {output_prefix}")
    return biogrid_query_target


In [11]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
pairs_df = pd.read_excel(excel_file)

# Assume BioGRID dataframe already loaded once
# biogrid = pd.read_csv("path_to_biogrid_file.txt", sep="\t")

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_biogrid_target1_query_for_pair(
        main_csv_path=main_file,
        biogrid_df=biogrid,
        output_prefix=SL_pair
    )


✅ BioGRID target1–query interactions saved for MSH6_WRN
✅ BioGRID target1–query interactions saved for CDH1_ROS1
✅ BioGRID target1–query interactions saved for ATM_ATR
✅ BioGRID target1–query interactions saved for BRCA1_ATR
✅ BioGRID target1–query interactions saved for CCNE1_PKMYT1
✅ BioGRID target1–query interactions saved for FBXW7_PKMYT1
✅ BioGRID target1–query interactions saved for BRCA1_PARG
✅ BioGRID target1–query interactions saved for BRCA1_USP1
✅ BioGRID target1–query interactions saved for BRCA1_POLQ
✅ BioGRID target1–query interactions saved for KRAS_PLK1


# BIOMARKER TSG or Oncogenes

In [14]:
def run_biomarker_cgc_classification_for_pair(main_csv_path: str, cgc_df: pd.DataFrame, output_prefix: str):
    """
    Classify biomarker genes as TSG (1), oncogene (0), or NA using Cancer Gene Census data.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATR_ATM_main.csv).
    cgc_df : pd.DataFrame
        Cancer Gene Census dataframe with columns ['Gene Symbol', 'Role in Cancer'].
    output_prefix : str
        Prefix for output file (e.g. "ATR_ATM").

    Returns
    -------
    biomarker_type : pd.DataFrame
        DataFrame with biomarker and TSG_Label classification.
    """
    # Load SL dataset
    main_csv = pd.read_csv(main_csv_path, low_memory=False)

    # Ensure consistent types
    main_csv['Biomarker'] = main_csv['Biomarker'].astype(str)
    cgc_df['Gene Symbol'] = cgc_df['Gene Symbol'].astype(str)

    # Build role mapping
    role_mapping = cgc_df.set_index('Gene Symbol')['Role in Cancer'].to_dict()

    # Classification function
    def classify_gene(symbol):
        role = role_mapping.get(symbol, None)
        if role:
            if 'TSG' in role:
                return 1
            elif 'oncogene' in role:
                return 0
        return np.nan

    # Apply to biomarkers
    main_csv['TSG_Label'] = main_csv['Biomarker'].apply(classify_gene)

    # Save results
    biomarker_type = main_csv[['Biomarker', 'TSG_Label']]
    biomarker_type.to_csv(
        f'feature_output/biomarker_type_{output_prefix}.csv',
        index=False
    )

    print(f"✅ Biomarker classification saved for {output_prefix}")
    return biomarker_type


In [15]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
pairs_df = pd.read_excel(excel_file)

# Load CGC once
cgc_df = pd.read_csv("/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/Cancer_gene_census_data.csv")

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_biomarker_cgc_classification_for_pair(
        main_csv_path=main_file,
        cgc_df=cgc_df,
        output_prefix=SL_pair
    )


✅ Biomarker classification saved for MSH6_WRN
✅ Biomarker classification saved for CDH1_ROS1
✅ Biomarker classification saved for ATM_ATR
✅ Biomarker classification saved for BRCA1_ATR
✅ Biomarker classification saved for CCNE1_PKMYT1
✅ Biomarker classification saved for FBXW7_PKMYT1
✅ Biomarker classification saved for BRCA1_PARG
✅ Biomarker classification saved for BRCA1_USP1
✅ Biomarker classification saved for BRCA1_POLQ
✅ Biomarker classification saved for KRAS_PLK1
